# 1. Schema & Nullability Testing

In [0]:
from pyspark.sql.functions import col, count

# 1. DEFINE VARIABLES (Dynamic Approach)
CATALOG_NAME = "vstone_catalog"
SCHEMA_NAME = "silver"
FULL_PATH = f"{CATALOG_NAME}.{SCHEMA_NAME}"

print(f"🚀 STARTING DYNAMIC TESTING FOR: {FULL_PATH}")
print("="*60)

# 2. GET TABLE LIST DYNAMICALLY (No Hardcoding)
# Ye command aapke schema se saari tables ki list nikaal degi
tables_list = [row['tableName'] for row in spark.sql(f"SHOW TABLES IN {FULL_PATH}").collect()]

# 3. LOOP THROUGH EACH TABLE FOR TESTING
for t_name in tables_list:
    full_table_path = f"{FULL_PATH}.{t_name}"
    
    try:
        # Table load karein
        df = spark.table(full_table_path)
        row_count = df.count()
        
        print(f"\n📊 TABLE: {t_name.upper()}")
        print(f"   - Total Rows: {row_count:,}")

        # --- TEST 1: DATA AVAILABILITY ---
        if row_count > 0:
            print("   ✅ Status: Passed (Data exists)")
        else:
            print("   ⚠️ Status: Warning (Table is empty)")

        # --- TEST 2: NULL INTEGRITY (Dynamic Column Detection) ---
        # Agar 'listing_id' hai toh use check karein, nahi toh pehle column ko
        check_col = "listing_id" if "listing_id" in df.columns else df.columns[0]
        nulls = df.filter(col(check_col).isNull()).count()
        
        if nulls == 0:
            print(f"   ✅ Integrity: 0 NULLs in '{check_col}'")
        else:
            print(f"   ❌ Integrity: Found {nulls} NULLs in '{check_col}'!")

        # --- TEST 3: QUARANTINE LOGIC CHECK ---
        # Sirf quarantine tables ke liye metadata check karein
        if "quarantine" in t_name.lower():
            if "quarantine_reason" in df.columns:
                print("   ✅ Audit: 'quarantine_reason' column is present.")
            else:
                print("   ❌ Audit: Missing 'quarantine_reason' in Quarantine table!")

    except Exception as e:
        print(f"   ❌ ERROR: Could not process {t_name}: {str(e)}")

print("\n" + "="*60 + "\n✨ ALL TABLES TESTED SUCCESSFULLY ✨")

# 2. Business Logic Testing (Currency & Normalization)

In [0]:
from pyspark.sql.functions import col, abs as spark_abs

# =============================================================
# 🔍 TEST 3: CURRENCY NORMALIZATION MATH CHECK (FIXED)
# =============================================================
print("\n🔍 STARTING TEST 3: CURRENCY NORMALIZATION ACCURACY")

# 1. Validation Logic using Spark
tolerance = 0.01
math_errors = df_silver.filter(col("price_rub") > 0) \
    .withColumn("diff", spark_abs(col("price_usd") - (col("price_rub") / 82.5))) \
    .filter(col("diff") > tolerance)

error_count = math_errors.count()

# 2. Results Output
if error_count == 0:
    print(f"✅ PASSED: Currency Normalization is 100% accurate.")
    
    # Ek sample row pick karein manual verification ke liye
    sample = df_silver.select("price_rub", "price_usd").filter(col("price_rub") > 0).first()
    
    # FIX: Python ka native format use kar rahe hain calculation ke liye
    rub_val = float(sample['price_rub'])
    usd_val = float(sample['price_usd'])
    manual_calc = rub_val / 82.5
    
    print(f"   Sample Verification Details:")
    print(f"   - Input RUB: {rub_val:,.2f}")
    print(f"   - Converted USD (in Table): {usd_val:,.2f}")
    print(f"   - Manual Formula Check: {rub_val:,.2f} / 82.5 = {manual_calc:,.2f}")
else:
    print(f"❌ FAILED: Found {error_count} records with calculation mismatch!")
    math_errors.select("listing_id", "price_rub", "price_usd", "diff").show(5)

print("-" * 50)

# 3. Bronze vs. Silver Reconciliation Script

In [0]:
# 1. SET GLOBAL CATALOG/SCHEMA VARIABLES
CATALOG_NAME = "vstone_catalog"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

print(f"📊 DYNAMIC MASTER AUDIT: {CATALOG_NAME} ECOSYSTEM")
print("="*80)

def get_table_counts(schema_name, filter_keyword=None, exclude_keyword=None):
    """Dynamically fetches table names from catalog and returns total count."""
    tables_df = spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{schema_name}")
    table_list = [row['tableName'] for row in tables_df.collect()]
    
    total_count = 0
    scanned_tables = []
    
    for t in table_list:
        # Filtering logic based on table names from your schema
        if filter_keyword and filter_keyword not in t:
            continue
        if exclude_keyword and exclude_keyword in t:
            continue
            
        full_path = f"{CATALOG_NAME}.{schema_name}.{t}"
        count = spark.table(full_path).count()
        total_count += count
        scanned_tables.append(f"{t} ({count:,})")
    
    return total_count, scanned_tables

try:
    # A. DYNAMIC BRONZE SCAN (All tables in bronze schema)
    # Based on your image: listings_csv, listings_json, listings_xml, etc.
    total_bronze, bronze_list = get_table_counts(BRONZE_SCHEMA)
    
    # B. DYNAMIC QUARANTINE SCAN (Only tables with 'quarantine' in name)
    # Based on your image: geography_quarantine, listings_main_quarantine, etc.
    total_quarantine, quarantine_list = get_table_counts(SILVER_SCHEMA, filter_keyword="quarantine")
    
    # C. DYNAMIC SILVER CLEAN SCAN (Tables in silver WITHOUT 'quarantine')
    # Based on your image: car_catalog_silver, geography_silver, listings_silver_merged
    total_silver, silver_list = get_table_counts(SILVER_SCHEMA, exclude_keyword="quarantine")

    # D. FINAL RECONCILIATION LOGIC
    # Duplicates are the gap between input and processed output
    duplicates = total_bronze - (total_silver + total_quarantine)

    # --- FINAL AUDIT REPORT ---
    print(f"📥 TOTAL BRONZE (Scanned {len(bronze_list)} Tables): {total_bronze:,}")
    print(f"⚠️ TOTAL QUARANTINE (Scanned {len(quarantine_list)} Tables): {total_quarantine:,}")
    print(f"✅ TOTAL SILVER CLEAN (Scanned {len(silver_list)} Tables): {total_silver:,}")
    print(f"♻️ CALCULATED DUPLICATES: {duplicates:,}")
    print("-" * 50)
    
    # Validation
    if total_bronze == (total_silver + total_quarantine + duplicates):
        print("🌟 DYNAMIC RECONCILIATION SUCCESS: 100% Data Accounted For!")
    else:
        print("❌ AUDIT DISCREPANCY: Please check pipeline logs.")

except Exception as e:
    print(f"❌ DYNAMIC AUDIT ERROR: {str(e)}")

print("="*80)

# 4. Quarantine Audit (Data Quality)

In [0]:
# --- TEST 6: QUARANTINE TABLE AUDIT ---
print("\n🔍 TEST 6: Quarantine Table Validation")
df_quarantine = spark.table("vstone_catalog.silver.listings_main_quarantine")

q_count = df_quarantine.count()
print(f"Total records in Quarantine: {q_count}")

if q_count > 0:
    print("Sample Quarantine Reasons:")
    df_quarantine.groupBy("quarantine_reason").count().show()

# 5. ACID & Time Travel Testing

In [0]:
# --- ENHANCED TEST 7: ACID VERSIONING & AUDIT TRAIL ---
print("\n🔍 TEST 7: Detailed Delta History & Audit Trail")

# 1. Variable define karein taaki code dynamic rahe
TABLE_PATH = "vstone_catalog.silver.listings_silver_merged"

try:
    # History fetch karein
    history_df = spark.sql(f"DESCRIBE HISTORY {TABLE_PATH}")
    
    # Versions count karein
    versions = history_df.count()

    if versions > 1:
        print(f"✅ ACID Verified: Table has {versions} versions tracked.")
        
        # 2. Sirf top 3 recent actions dikhaayein (What happened?)
        print("\n📜 Recent Transaction Summary:")
        history_df.select("version", "timestamp", "operation", "operationParameters.mode") \
                  .orderBy(col("version").desc()) \
                  .show(3, truncate=False)

        # 3. Time Travel Verification (Optional but very detailed)
        # Hum check kar rahe hain ki kya hum Version 0 (Raw) par wapas ja sakte hain
        print(f"🕒 Checking Time Travel capability for {TABLE_PATH}...")
        v0_count = spark.read.option("versionAsOf", 0).table(TABLE_PATH).count()
        print(f"✅ Time Travel Success: Version 0 had {v0_count:,} records.")

    else:
        print("⚠️ Warning: Only 1 version found. Acid history might be limited.")

except Exception as e:
    print(f"❌ History Audit Failed: {str(e)}")

print("-" * 60)